In [1]:
import pandas as pd

url = "https://docs.google.com/spreadsheets/d/17yC9PcTkC3gY5orbQ5VQXeBDOStMx2Dfpo53cMZTn5k/export?format=xlsx"

df_1 = pd.read_excel(url, sheet_name="Tipos de vigas")
df_2 = pd.read_excel(url, sheet_name="Capacidade das fôrmas")



print(df_1)
print(df_2)

   Tipo  Tamanho (m)  Demanda
0     1         1.22       24
1     2         1.45       60
2     3         2.35       56
3     4         2.50       72
4     5         2.65       16
5     6         2.95       17
6     7         3.30       12
   Tipo de fôrma  Capacidade (m)
0              1           11.95
1              2           11.95
2              3           11.95
3              4           11.95
4              5           11.95
5              6           11.95
6              7            5.95


In [4]:
import random

def grasp_csp(vigas_demand, tamanho_forma, max_iter=100, alpha=0.3):
    # 1. Expandir a demanda em uma lista plana de peças
    pecas_iniciais = []
    for tamanho, demanda in vigas_demand.items():
        pecas_iniciais.extend([tamanho] * demanda)
        
    melhor_solucao = []
    menor_numero_formas = float('inf')

    # Loop principal do GRASP
    for iteracao in range(max_iter):
        pecas = pecas_iniciais.copy()
        solucao_atual = []
        
        # Fase 1: Construção Gulosa Randomizada
        while pecas:
            forma_atual = []
            espaco_livre = tamanho_forma
            
            while True:
                # Filtrar apenas as peças que ainda cabem na fôrma atual
                candidatos = [p for p in pecas if p <= espaco_livre]
                
                if not candidatos:
                    break # Fôrma atual não cabe mais nada
                    
                # Definir a qualidade para a RCL (peças maiores são melhores)
                max_val = max(candidatos)
                min_val = min(candidatos)
                
                # O limite define o quão rigorosa é a seleção (baseado no Alpha)
                limite = max_val - alpha * (max_val - min_val)
                rcl = [p for p in candidatos if p >= limite]
                
                # Escolha aleatória dentro da lista restrita de candidatos excelentes
                escolha = random.choice(rcl)
                
                forma_atual.append(escolha)
                pecas.remove(escolha)
                espaco_livre -= escolha
                
            solucao_atual.append(forma_atual)
            
        # Fase 2: Busca Local (Conceitual - simplificada)
        # Uma busca local agressiva tentaria esvaziar a fôrma com a menor soma
        solucao_atual = busca_local_simples(solucao_atual, tamanho_forma)
        
        # Avaliar se encontramos uma solução melhor (menos fôrmas)
        if len(solucao_atual) < menor_numero_formas:
            menor_numero_formas = len(solucao_atual)
            melhor_solucao = solucao_atual
            
    return melhor_solucao

def busca_local_simples(solucao, tamanho_forma):
    """
    Tenta pegar as peças da fôrma com maior desperdício e realocá-las.
    """
    # Ordenar as fôrmas da mais cheia para a mais vazia
    solucao = sorted(solucao, key=lambda f: sum(f), reverse=True)
    forma_mais_vazia = solucao[-1]
    
    # Tenta alocar as peças da fôrma mais vazia nas sobras das outras
    restantes = []
    for peca in forma_mais_vazia:
        alocada = False
        for forma in solucao[:-1]:
            if sum(forma) + peca <= tamanho_forma:
                forma.append(peca)
                alocada = True
                break
        if not alocada:
            restantes.append(peca)
            
    # Se conseguiu realocar todas as peças, elimina a fôrma mais vazia
    if not restantes:
        return solucao[:-1]
    return solucao

# Dados da Instância
vigas = { 1.22: 24, 1.45: 60, 2.35: 56, 2.50: 72, 2.65: 16, 2.95: 17, 3.30: 12 }
tamanho_forma = 11.95

# Rodando o algoritmo GRASP
solucao_final = grasp_csp(vigas, tamanho_forma, max_iter=500, alpha=0.4)

# Resultados
desperdicio = sum(tamanho_forma - sum(f) for f in solucao_final)
print(f"Total de fôrmas utilizadas: {len(solucao_final)}")
print(f"Desperdício total: {desperdicio:.2f}m")

Total de fôrmas utilizadas: 49
Desperdício total: 25.52m
